# Centipede: training on Kaggle

This notebook trains the centipede on Kaggle's GPUs and keeps every result.
It clones the public repositories of Centipede, RL_lib, and MujocoReplay,
installs them, and runs `python -m centipede` with a configuration file from
the repository, one run per GPU.

**Before running**, in the notebook's side panel:

- **Accelerator:** GPU T4 x2 (two runs at once) or GPU T4 (one run).
- **Internet:** on.

**To train for hours**, do not use Run all in the open editor, which stops
when the browser session ends. Choose **Save Version → Save & Run All
(Commit)**: Kaggle runs the whole notebook in the background, for up to 12
hours, and keeps everything written in `/kaggle/working` as the version's
output. Each run folder (configuration, log, checkpoints, report, recordings)
is in `/kaggle/working/runs`.

**Sessions and continuing.** A run longer than one session stops cleanly, with
a checkpoint, before the time limit set below. To continue it, open the
notebook again, add the previous version's output as input (**Add Input → Your
Work → this notebook**), set `CONTINUE_RUNS` to the run folders as they appear
under `/kaggle/input`, and save a new version. The run folder is copied into
`/kaggle/working/runs`, continues from its latest checkpoint, and is saved
again with the new version's output.

The first step of every session compiles the GPU kernels, about a minute.

In [ ]:
# Settings: the only cell to edit.

# The configuration file of a new run, from the repository's configs folder.
CONFIGURATION = "configs/training.toml"
# New runs: one seed per GPU. [1] trains one run; [1, 2] trains two at once,
# with the same settings and different seeds, on the two T4s.
NEW_RUN_SEEDS = [1, 2]
# Runs to continue instead, at most one per GPU: their folders under
# /kaggle/input, for example
# ["/kaggle/input/centipede-training/runs/2026-10-08_0900_training_seed1"].
# When this list is not empty, NEW_RUN_SEEDS is ignored.
CONTINUE_RUNS = []
# Each session stops cleanly before this many hours, leaving time for setup
# and for Kaggle to save the output within its 12-hour limit.
TIME_LIMIT_HOURS = 11.25
# The Centipede branch to train with.
BRANCH = "main"

In [ ]:
# The machine: GPUs, driver, CPU cores, and Python version.
!nvidia-smi --query-gpu=index,name,memory.total,driver_version --format=csv
!nproc
!python --version

In [ ]:
# Fetch the project and the two sibling packages it installs; BRANCH is set
# in the settings cell. They go to /tmp, so that the version's output holds
# only the runs and their logs.
OWNER = "https://github.com/DrippingSoup22"
!rm -rf /tmp/centipede /tmp/RL_lib /tmp/MujocoReplay
!git clone --depth 1 --branch {BRANCH} {OWNER}/centipede-project.git /tmp/centipede
!git clone --depth 1 {OWNER}/RL_lib.git /tmp/RL_lib
!git clone --depth 1 {OWNER}/MujocoReplay.git /tmp/MujocoReplay
%cd /tmp/centipede
!git log -1 --format="Centipede commit %h: %s"
!git -C /tmp/RL_lib log -1 --format="RL_lib commit %h: %s"
!git -C /tmp/MujocoReplay log -1 --format="MujocoReplay commit %h: %s"

In [ ]:
# Install RL_lib (the learning algorithms) and MujocoReplay (which writes the
# replay recordings), then the package with the GPU backend. Kaggle
# already provides PyTorch with CUDA, so it is not downloaded again.
!pip install --quiet -e /tmp/RL_lib -e /tmp/MujocoReplay
!pip install --quiet -e ".[gpu]"

In [ ]:
# The versions in use, and the GPU as PyTorch and Warp see it.
import mujoco
import mujoco_warp
import torch
import warp

print("MuJoCo", mujoco.__version__, "| MuJoCo Warp", mujoco_warp.__version__)
print("Warp", warp.__version__, "| PyTorch", torch.__version__)
print(
    "GPU:",
    torch.cuda.get_device_name(),
    "| compute capability",
    torch.cuda.get_device_capability(),
    "|",
    torch.cuda.device_count(),
    "visible",
)
assert torch.cuda.get_device_capability() >= (7, 0), "Choose the T4 accelerator"

In [ ]:
# One session file per run: a new run from the configuration with its seed,
# or a continuing run copied from the input. Runs are written to
# /kaggle/working/runs, so they are kept in the version's output.
import shutil
import subprocess
import tomllib
from pathlib import Path

import tomli_w
import torch

RUNS = Path("/kaggle/working/runs")
SESSIONS = Path("/kaggle/working/sessions")
RUNS.mkdir(exist_ok=True)
SESSIONS.mkdir(exist_ok=True)
gpu_count = torch.cuda.device_count()

session_files = []
if CONTINUE_RUNS:
    assert len(CONTINUE_RUNS) <= gpu_count, f"at most {gpu_count} runs at once"
    for source in map(Path, CONTINUE_RUNS):
        target = RUNS / source.name
        if not target.exists():
            shutil.copytree(source, target)
        session = {
            "run": {
                "mode": "train",
                "continue_from": target.as_posix(),
                "time_limit_hours": TIME_LIMIT_HOURS,
            }
        }
        path = SESSIONS / f"{source.name}.toml"
        path.write_text(tomli_w.dumps(session), encoding="utf-8")
        session_files.append(path)
else:
    assert len(NEW_RUN_SEEDS) <= gpu_count, f"at most {gpu_count} runs at once"
    base = tomllib.loads(Path(CONFIGURATION).read_text(encoding="utf-8"))
    for key in ("runs_folder", "time_limit_hours"):
        assert key not in base["run"], f"this notebook sets [run] {key}"
    for seed in NEW_RUN_SEEDS:
        session = {**base, "run": dict(base["run"])}
        session["run"]["seed"] = seed
        session["run"]["name"] = f"{base['run']['name']}_seed{seed}"
        session["run"]["runs_folder"] = RUNS.as_posix()
        session["run"]["time_limit_hours"] = TIME_LIMIT_HOURS
        path = SESSIONS / f"{session['run']['name']}.toml"
        path.write_text(tomli_w.dumps(session), encoding="utf-8")
        session_files.append(path)

for path in session_files:
    print(f"--- {path}")
    print(path.read_text(encoding="utf-8"))

In [ ]:
# Train: one process per run, each on its own GPU, logging to
# /kaggle/working/logs. Every five minutes the cell prints each run's latest
# progress line, with the time its remaining cycles will take.
import os
import time

LOGS = Path("/kaggle/working/logs")
LOGS.mkdir(exist_ok=True)
processes = []
for gpu, path in enumerate(session_files):
    log_path = LOGS / f"{path.stem}.log"
    environment = os.environ | {"CUDA_VISIBLE_DEVICES": str(gpu)}
    log = open(log_path, "w", encoding="utf-8")
    command = ["python", "-u", "-m", "centipede", str(path)]
    processes.append((path.stem, subprocess.Popen(command, stdout=log, stderr=subprocess.STDOUT, env=environment), log_path))
    print(f"GPU {gpu}: {path.stem}, log {log_path}")


def last_progress_line(log_path):
    lines = [line for line in log_path.read_text(encoding="utf-8").splitlines() if line.strip()]
    progress = [line for line in lines if line.startswith(("cycle", "Stopped", "Finished", "Traceback"))]
    return (progress or lines or ["starting ..."])[-1]


while any(process.poll() is None for _, process, _ in processes):
    time.sleep(300)
    print(time.strftime("%H:%M"))
    for name, _, log_path in processes:
        print(f"  {name}: {last_progress_line(log_path)}")

for name, process, log_path in processes:
    print(f"
=== {name}: exit code {process.returncode}")
    print("
".join(log_path.read_text(encoding="utf-8").splitlines()[-15:]))

In [ ]:
# What the output holds: each run folder, with its checkpoints and report.
!find /kaggle/working/runs -maxdepth 2 | sort
!du -sh /kaggle/working/runs/*